In [164]:
import numpy as np
from scipy.optimize import least_squares
from scipy.special import gamma
from dataclasses import dataclass
import pandas as pd

In [165]:
country=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\01_baseline_country_table.csv')
trade=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\02_baseline_trade_table.csv')
parameter=pd.read_csv(r'E:\zotero\部分论文复刻\JIE2019资本积累和动态效应检验\自我复刻代码\CSV_数据处理与校准输出\03_baseline_parameter_table.csv')
country=country.merge(parameter, on='country')

In [166]:
# 全局标量参数
@dataclass(frozen=True)
class ModelParams:
    theta: float = 4.0
    eta: float = 2.0
    beta: float = 0.96
    sigma: float = 0.5
    delta: float = 0.06
    lamb: float = 0.76
    alpha: float = 0.33

    @property
    def chi(self):
        return self.delta ** (1 - self.lamb)

    @property
    def gamma_term(self):
        return gamma(1 + (1 / self.theta) * (1 - self.eta)) ** (1 / (1 - self.eta))
        

    @property
    def Alpha(self):
        return self.alpha ** (-self.alpha) * (1 - self.alpha) ** (self.alpha - 1)
    
    @property
    def phi_state_1(self):
        return 1/self.lamb
    
    @property
    def phi_state_2(self):
        return self.delta-1/self.lamb

    @property
    def q(self):
        return 1/self.beta-1
p: ModelParams = ModelParams()

In [167]:
@dataclass(frozen=True)
class CountryParams:
    nu_m: np.ndarray
    nu_c: np.ndarray
    nu_x: np.ndarray
    T_m: np.ndarray
    A_c: np.ndarray
    A_x: np.ndarray
    F: np.ndarray
    L: np.ndarray
    K: np.ndarray
    @property
    def N(self):
        return len(self.nu_m)

cp: CountryParams = CountryParams(
    nu_m=country["nu_m0"].values,
    nu_c=country["nu_c0"].values,
    nu_x=country["nu_x0"].values,
    T_m=country["T_m0"].values,
    A_c=country["A_c0"].values,
    A_x=country["A_x0"].values,
    F=country["F"].values,
    L=country["L0"].values,
    K=country["K0"].values,
)

In [168]:
#这是第一版写法，这是不对的，在数值中，特别是多国模型中，能用数组就用数组，因此MATLAB还是非常方便推荐的
# def u_j(r_j,w_j,P_mj,p,nu):
#     return r_j/(p.alpha*nu)**((p.alpha*nu))\
#             *w_j/((1-p.alpha)*nu)**((1-p.alpha)*nu) \
#             *P_mj/(1-nu)**(1-nu)

def unit_cost(r,w,P_m,p,nu):
    return (r/(p.alpha*nu))**(p.alpha*nu) \
        *(w/((1-p.alpha)*nu))**((1-p.alpha)*nu) \
        *(P_m/(1-nu))**(1-nu)

In [169]:
# 使用残差法求解方程，可以用不动点理解，但实际上用官方的包来写
def solve_prices_given_wage(wage,trade_cost,p,cp,max_iter=100000,tol=1e-6):
    P_x=np.ones(cp.N)
    P_m=np.ones(cp.N)
    for i in range(max_iter):
        r=(p.phi_state_1/p.beta+p.phi_state_2)*P_x
        u_m=unit_cost(r,wage,P_m,p,cp.nu_m)

        P_m_new=p.gamma_term*(np.sum((u_m*trade_cost)**(-p.theta)*cp.T_m,axis=1))**(-1/p.theta) #更新
        P_x_mew=unit_cost(r,wage,P_m,p,cp.nu_x)/cp.A_x

        distance=max(
            np.max(np.abs(P_m_new/P_m-1)),
            np.max(np.abs(P_x_mew/P_x-1))
                   )
        P_m=P_m_new
        P_x=P_x_mew
        if distance<tol :
            break
    r=(p.phi_state_1/p.beta+p.phi_state_2)*P_x
    P_c=(1/cp.A_c)*unit_cost(r,wage,P_m,p,cp.nu_c)
    return P_x,P_m,P_c,r

In [170]:
# 使用残差法求解方程，可以用不动点理解，但实际上用官方的包来写

def price_residual(vars, wage, trade_cost, p, cp):
    #拼凑未知变量var
    P_m = vars[:cp.N]
    P_x = vars[cp.N:2*cp.N]
    r = vars[2*cp.N:3*cp.N]


    u_m = unit_cost(r, wage, P_m, p, cp.nu_m)

    P_m_implied = (
        p.gamma_term
        * np.sum(
            cp.T_m * (u_m * trade_cost) ** (-p.theta),
            axis=1
        ) ** (-1 / p.theta)
    )
    P_x_implied = unit_cost(r, wage, P_m, p, cp.nu_x)/ cp.A_x

    # 资本租金
    r_implied = (p.phi_state_1 / p.beta + p.phi_state_2) * P_x

    return np.r_[
        P_m_implied / P_m - 1,
        P_x_implied / P_x - 1,
        r_implied / r - 1,
    ]


def solve_prices_given_wage_residual(wage, trade_cost, p, cp):
    P_m0 = np.ones(cp.N)
    P_x0 = np.ones(cp.N)
    r0 = (p.phi_state_1 / p.beta + p.phi_state_2) * P_x0

    x0 = np.r_[P_m0, P_x0, r0]

    fit = least_squares(
        price_residual,
        x0=x0,
        args=(wage, trade_cost, p, cp),
        ftol=1e-10,
        xtol=1e-10,
        gtol=1e-10,
        max_nfev=50_000,
    )

    if not fit.success:
        raise RuntimeError(f"价格系统求解失败：{fit.message}")

    x = fit.x

    P_m = x[:cp.N]
    P_x = x[cp.N:2*cp.N]
    r = x[2*cp.N:3*cp.N]

    P_c = unit_cost(r, wage, P_m, p, cp.nu_c) / cp.A_c

    return P_x, P_m, P_c, r

In [171]:
def compute_trade_share(wage,p,cp,trade_cost,r,P_m):
    u_m=unit_cost(r,wage,P_m,p,cp.nu_m)
    numerator=(u_m*trade_cost)**(-p.theta)*cp.T_m
    denominator=np.sum((u_m*trade_cost)**(-p.theta)*cp.T_m,axis=1)
    return numerator / denominator[:, None]
# ij代表了i买j的比例。

In [172]:
### 简单到复杂，整合函数代码，减少内部反复调用和重复计算
@dataclass(frozen=True)
class Quantities:
    K: np.ndarray
    X: np.ndarray
    C: np.ndarray
    Y_m: np.ndarray

    L_c: np.ndarray
    L_x: np.ndarray
    L_m: np.ndarray

    K_c: np.ndarray
    K_x: np.ndarray
    K_m: np.ndarray

    M_c: np.ndarray
    M_x: np.ndarray
    M_m: np.ndarray
    M_use: np.ndarray

    deficit: np.ndarray

def compute_quantity(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    #根据猜测的工资和四组价格，求出来C和KXM及其分配等13个变量
    K=p.alpha/(1-p.alpha)*wage * cp.L / r
    X=K*p.delta
    C=(r*K+wage*cp.L+cp.F-P_x*X)/P_c #根据均衡条件，CP.F也就是净出口为qA

    A1=(1-cp.nu_c)*P_c*C
    A2=(1-cp.nu_x)*P_x*X
    B1=trade_share.T@(A1+A2)
    B2=np.diag(P_m)-trade_share.T@np.diag((1-cp.nu_m)*P_m)
    Y_m=np.linalg.solve(B2,B1)

    L_c=(1-p.alpha)*cp.nu_c*P_c*C/wage
    L_x=(1-p.alpha)*cp.nu_x*P_x*X/wage
    L_m=(1-p.alpha)*cp.nu_m*P_m*Y_m/wage

    K_c=p.alpha/(1-p.alpha)*L_c*wage/r
    K_x=p.alpha/(1-p.alpha)*L_x*wage/r
    K_m=p.alpha/(1-p.alpha)*L_m*wage/r      

    M_c=(1-cp.nu_c)*P_c*C/ P_m
    M_x=(1-cp.nu_x)*P_x*X/ P_m
    M_m=(1-cp.nu_m)*P_m*Y_m/ P_m
    
    M_use=M_c+M_x+M_m

    deficit=P_m*M_use-P_m*Y_m
    return Quantities(
    K=K,
    X=X,
    C=C,
    Y_m=Y_m,
    L_c=L_c,
    L_x=L_x,
    L_m=L_m,
    K_c=K_c,
    K_x=K_x,
    K_m=K_m,
    M_c=M_c,
    M_x=M_x,
    M_m=M_m,
    M_use=M_use,
    deficit=deficit,
)

In [173]:
def solve_K(r,wage,p,cp):
    return p.alpha/(1-p.alpha)*wage * cp.L / r
def solve_invest(r,wage,p,cp):
    return solve_K(r,wage,p,cp)*p.delta
def solve_consum(r,wage,p,cp,P_x,P_c):
    right=r*solve_K(r,wage,p,cp)+wage*cp.L+cp.F
    left=P_x*solve_invest(r,wage,p,cp)
    return (right-left)/P_c
    
def solve_Y_m(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    A1=(1-cp.nu_c)*P_c*solve_consum(r,wage,p,cp,P_x,P_c)
    A2=(1-cp.nu_x)*P_x*solve_invest(r,wage,p,cp)
    
    B1=trade_share.T@(A1+A2)
    B2=np.diag(P_m)-trade_share.T@np.diag((1-cp.nu_m)*P_m)
    return np.linalg.solve(B2,B1)
def solve_L_c(r,wage,p,cp,P_x,P_c):
    return (1-p.alpha)*cp.nu_c*P_c*solve_consum(r,wage,p,cp,P_x,P_c)/wage

def solve_L_x(r,wage,p,cp,P_x,P_c):
    return (1-p.alpha)*cp.nu_x*P_x*solve_invest(r,wage,p,cp)/wage

def solve_L_m(r,wage,p,cp,P_x,P_c):
    return cp.L-solve_L_c(r,wage,p,cp,P_x,P_c)-solve_L_x(r,wage,p,cp,P_x,P_c)

def solve_K_c(r,wage,p,cp,P_x,P_c):
    return p.alpha/(1-p.alpha)*solve_L_c(r,wage,p,cp,P_x,P_c)*wage/r

def solve_K_x(r,wage,p,cp,P_x,P_c):
    return p.alpha/(1-p.alpha)*solve_L_x(r,wage,p,cp,P_x,P_c)*wage/r

def solve_K_m(r,wage,p,cp,P_x,P_c):
    return p.alpha/(1-p.alpha)*solve_L_m(r,wage,p,cp,P_x,P_c)*wage/r

def solve_M_c(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    return (1-cp.nu_c)*P_c*solve_consum(r,wage,p,cp,P_x,P_c)/ P_m

def solve_M_x(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    return (1-cp.nu_x)*P_x*solve_invest(r,wage,p,cp)/ P_m

def solve_M_m(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    return (1-cp.nu_m)*P_m*solve_Y_m(r,wage,p,cp,P_x,P_c,P_m,trade_share)/ P_m

def solve_M(r,wage,p,cp,P_x,P_c,P_m,trade_share):
    return solve_M_c(r,wage,p,cp,P_x,P_c,P_m,trade_share)+solve_M_x(r,wage,p,cp,P_x,P_c,P_m,trade_share)+solve_M_m(r,wage,p,cp,P_x,P_c,P_m,trade_share)

In [174]:
def wage_residual(log_wage,p,cp,trade_cost):
    wage_raw=np.exp(log_wage)
    wage = wage_raw / np.sum(wage_raw * cp.L)
    P_x, P_m, P_c, r=solve_prices_given_wage(wage,trade_cost,p,cp)
    trade_share=compute_trade_share(wage,p,cp,trade_cost,r,P_m)
    quantity=compute_quantity(r,wage,p,cp,P_x,P_c,P_m,trade_share)
    residual=(cp.F-quantity.deficit)/np.maximum(wage*cp.L,1e-12) #防止除以0 maximum是主元素比较
    return residual

In [175]:
def solve_steady_state(p,cp,trade_cost):
    log_wage0=np.zeros(cp.N)
    fit=least_squares(
        wage_residual,
        x0=log_wage0,
        args=(p,cp,trade_cost),
        ftol=1e-8,
        xtol=1e-8,
        gtol=1e-8,
        max_nfev=50_000,
    )
    if not fit.success:
        raise RuntimeError(f"工资系统求解失败：{fit.message}")
    #这就算完了，然后重新还原一遍
    wage_raw = np.exp(fit.x)
    wage = wage_raw / np.sum(wage_raw * cp.L)

    P_x, P_m, P_c, r = solve_prices_given_wage_residual(
        wage, trade_cost, p, cp
    )

    ts = compute_trade_share(wage, p, cp, trade_cost, r, P_m)

    qnt = compute_quantity(
        r, wage, p, cp, P_x, P_c, P_m, ts
    )

    return {
        "wage": wage,
        "r": r,
        "P_x": P_x,
        "P_m": P_m,
        "P_c": P_c,
        "trade_share": ts,
        "quantity": qnt,
        "residual": wage_residual(fit.x, p, cp, trade_cost),
        "fit": fit,
    }

In [176]:
countries=country["country"].to_list()
observed_trade_share  = (
    trade
    .pivot(index="destination", columns="origin", values="trade_share")
    .reindex(index=countries, columns=countries)
    .to_numpy()
)
trade_costs= (
    trade
    .pivot(index="destination", columns="origin", values="trade_cost")
    .reindex(index=countries, columns=countries)
    .to_numpy()
)

In [177]:
ss = solve_steady_state(p, cp, trade_costs)

C:\Users\27968\AppData\Local\Temp\ipykernel_29224\3666107091.py:8: RuntimeWarning: invalid value encountered in power
  return (r/(p.alpha*nu))**(p.alpha*nu) \
C:\Users\27968\AppData\Local\Temp\ipykernel_29224\3666107091.py:10: RuntimeWarning: invalid value encountered in power
  *(P_m/(1-nu))**(1-nu)


In [178]:
ss

{'wage': array([1.41778487e-03, 1.06989559e-03, 1.07945798e-03, 1.89273785e-04,
        2.68817709e-04, 1.22249730e-03, 1.78871125e-03, 1.85498364e-04,
        6.26490340e-04, 4.18368422e-04, 1.00529402e-03, 1.35250920e-03,
        8.34190276e-04, 4.69834855e-04, 1.09885069e-03, 1.02267111e-03,
        9.84664915e-04, 5.38752141e-04, 4.05570552e-04, 3.35184673e-04,
        1.06228824e-04, 6.04060032e-05, 1.76750107e-03, 9.37633584e-04,
        7.94288818e-04, 5.87649831e-04, 6.27622396e-04, 9.64359512e-04,
        3.33671276e-04, 3.65905294e-04, 4.78765263e-04, 1.15652507e-03,
        2.66440438e-03, 4.77029928e-04, 5.04467323e-04, 2.74365486e-04,
        1.65179799e-04, 3.41346182e-04, 5.34934151e-04, 5.50837852e-04,
        1.30770191e-03, 4.60095396e-04, 5.82509043e-04, 1.58427208e-03]),
 'r': array([0.00184165, 0.00116463, 0.00106866, 0.00068566, 0.00079478,
        0.00130339, 0.00141166, 0.00079072, 0.00083314, 0.00083035,
        0.00122497, 0.00139782, 0.00086974, 0.00091254, 0

In [179]:
q = ss["quantity"]

print("外部清算残差:", np.max(np.abs(ss["residual"])))
print("贸易份额行和:", np.max(np.abs(ss["trade_share"].sum(axis=1) - 1)))
print("劳动清算:", np.max(np.abs(q.L_c + q.L_x + q.L_m - cp.L)))
print("资本清算:", np.max(np.abs(q.K_c + q.K_x + q.K_m - q.K)))
labor_error = q.L_c + q.L_x + q.L_m - cp.L
capital_error = q.K_c + q.K_x + q.K_m - q.K

print("劳动清算相对误差:", np.max(np.abs(labor_error) / np.maximum(cp.L, 1e-12)))
print("资本清算相对误差:", np.max(np.abs(capital_error) / np.maximum(q.K, 1e-12)))

外部清算残差: 3.024600284131649e-09
贸易份额行和: 7.771561172376096e-16
劳动清算: 0.0010997605102147645
资本清算: 0.00012707273452861045
劳动清算相对误差: 1.3775111085069577e-06
资本清算相对误差: 1.3775111083642273e-06


### 模型拟合度检验

In [180]:
qnt = ss["quantity"]

ss_country = pd.DataFrame({
    "country": countries,

    "wage_model": ss["wage"],
    "rent_model": ss["r"],
    "P_x_model": ss["P_x"],
    "P_m_model": ss["P_m"],
    "P_c_model": ss["P_c"],

    "K_model": qnt.K,
    "X_model": qnt.X,
    "C_model": qnt.C,
    "Y_m_model": qnt.Y_m,

    "L_c_model": qnt.L_c,
    "L_x_model": qnt.L_x,
    "L_m_model": qnt.L_m,

    "K_c_model": qnt.K_c,
    "K_x_model": qnt.K_x,
    "K_m_model": qnt.K_m,

    "deficit_model": qnt.deficit,
})

In [ ]:
check = country.merge(
    ss_country,
    on="country",
    how="left",
)

check["C_data"] = check["Con"] / check["pl_c"]
check["X_data"] = check["GFCF"] / check["pl_i"]
check["Y_m_data"] = check["GO_m"] / check["P_m"]

check["Px_over_Pm_data"] = check["pl_i"] / check["P_m"]
check["Pc_over_Pm_data"] = check["pl_c"] / check["P_m"]

check["Px_over_Pm_model"] = check["P_x_model"] / check["P_m_model"]
check["Pc_over_Pm_model"] = check["P_c_model"] / check["P_m_model"]

check["income_per_worker_data"] = (
    check["GDP"] / check["L0"] / check["pl_c"]
)

check["income_per_worker_model"] = (
    check["wage_model"] / check["P_c_model"] / (1 - check["alpha0"])
)

check["investment_rate_data"] = check["GFCF"] / check["GDP"]

check["investment_rate_model"] = (
    check["P_x_model"] * check["X_model"]
    / (check["wage_model"] * check["L0"] + check["rent_model"] * check["K_model"])
)

check["deficit_data"] = check["F"]

In [183]:
def fit_stats(df, model_col, data_col, log=True):
    x = df[model_col].to_numpy(dtype=float)
    y = df[data_col].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    if log:
        mask = mask & (x > 0) & (y > 0)
        x = np.log(x[mask])
        y = np.log(y[mask])
    else:
        x = x[mask]
        y = y[mask]

    error = x - y

    return {
        "model": model_col,
        "data": data_col,
        "n": len(x),
        "corr": np.corrcoef(x, y)[0, 1] if len(x) > 1 else np.nan,
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "max_abs_error": np.max(np.abs(error)),
    }

In [184]:
target_checks = [
    ("Px_over_Pm_model", "Px_over_Pm_data"),
    ("Pc_over_Pm_model", "Pc_over_Pm_data"),
    ("income_per_worker_model", "income_per_worker_data"),
]

target_report = pd.DataFrame([
    fit_stats(check, model_col, data_col, log=True)
    for model_col, data_col in target_checks
])

target_report

,model,data,n,corr,mae,rmse,max_abs_error
0,Px_over_Pm_model,Px_over_Pm_data,44,0.991381,0.063752,0.075242,0.284133
1,Pc_over_Pm_model,Pc_over_Pm_data,44,0.994414,0.261614,0.264644,0.404537
2,income_per_worker_model,income_per_worker_data,44,0.970113,26.860408,26.860678,27.216124


In [185]:
pi_model = ss["trade_share"]
pi_data = observed_trade_share

mask = (
    np.isfinite(pi_model)
    & np.isfinite(pi_data)
    & (pi_model > 0)
    & (pi_data > 0)
)

pi_log_error = np.log(pi_model[mask]) - np.log(pi_data[mask])

trade_share_report = {
    "n": int(mask.sum()),
    "corr_log": np.corrcoef(np.log(pi_model[mask]), np.log(pi_data[mask]))[0, 1],
    "mae_log": np.mean(np.abs(pi_log_error)),
    "rmse_log": np.sqrt(np.mean(pi_log_error ** 2)),
    "max_abs_level": np.max(np.abs(pi_model[mask] - pi_data[mask])),
}

trade_share_report

{'n': 1936,
 'corr_log': 0.9920618917291295,
 'mae_log': 0.223139359101811,
 'rmse_log': 0.2990016308868058,
 'max_abs_level': 0.15328173962164218}

In [186]:
moment_checks = [
    ("C_model", "C_data"),
    ("X_model", "X_data"),
    ("Y_m_model", "Y_m_data"),
    ("investment_rate_model", "investment_rate_data"),
]

moment_report = pd.DataFrame([
    fit_stats(check, model_col, data_col, log=(model_col != "investment_rate_model"))
    for model_col, data_col in moment_checks
])

moment_report

,model,data,n,corr,mae,rmse,max_abs_error
0,C_model,C_data,44,0.996595,58.701829,58.702047,59.111732
1,X_model,X_data,44,0.992561,27.330607,27.331548,27.979650
2,Y_m_model,Y_m_data,44,0.980623,26.060064,26.062675,26.989333
3,investment_rate_model,investment_rate_data,44,-0.158647,0.064939,0.084144,0.291627
